# PBMC preprocessing QC

This notebook is **read-only QC** for the frozen PBMC benchmark object. It does
not create or modify the benchmark population or representations.

Expected workflow:
1. healthy controls only;
2. donor cap at 500 cells using the legacy PBMC RNG behavior;
3. apply the 200-cell / 5-donor cell-type eligibility rule;
4. select 1,000 donor-aware HVGs;
5. compute 15-dimensional PCA and Harmony;
6. retain the source-provided `X_scVI` embedding without retraining it.

In [1]:
from pathlib import Path
import json

import anndata as ad
import pandas as pd
import yaml

def find_repo_root(start=Path.cwd()):
    start = Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / "src" / "scrna_benchmark").is_dir():
            return p
    raise RuntimeError("Run this notebook from inside the repository.")

REPO_ROOT = find_repo_root()
ADATA_PATH = REPO_ROOT / "data/PBMC_Stephenson/stephenson_benchmark_ready.h5ad"
SOURCE_PATH = REPO_ROOT / "data/PBMC_Stephenson/stephenson_2021_subsampled.h5ad"
QC_DIR = REPO_ROOT / "results/preprocessing/pbmc"
SUMMARY_PATH = QC_DIR / "preprocessing_summary.json"
print(REPO_ROOT)

/users/xchen5/scRNA-cross-donor-generalization


## Frozen-object invariants

In [2]:
if not ADATA_PATH.exists():
    raise FileNotFoundError(ADATA_PATH)
if not SUMMARY_PATH.exists():
    raise FileNotFoundError(SUMMARY_PATH)

adata = ad.read_h5ad(ADATA_PATH, backed="r")
with SUMMARY_PATH.open() as f:
    summary = json.load(f)

observed = {
    "cells": adata.n_obs,
    "HVGs": adata.n_vars,
    "donors": adata.obs["patient_id"].astype(str).nunique(),
    "cell_types": adata.obs["cell_type"].astype(str).nunique(),
    "sites": adata.obs["Site"].astype(str).nunique(),
    "PCA_dims": adata.obsm["X_pca"].shape[1],
    "Harmony_dims": adata.obsm["X_harmony"].shape[1],
    "scVI_dims": adata.obsm["X_scVI"].shape[1],
    "has_counts_layer": "counts" in adata.layers,
}
display(pd.DataFrame([observed]))

assert observed["cells"] == 10107
assert observed["HVGs"] == 1000
assert observed["donors"] == 23
assert observed["cell_types"] == 13
assert observed["sites"] == 2
assert observed["PCA_dims"] == 15
assert observed["Harmony_dims"] == 15
assert observed["scVI_dims"] > 0
assert not observed["has_counts_layer"], (
    "PBMC reuses the source-provided scVI embedding; a counts layer is not "
    "required by this preprocessing workflow."
)
print("All frozen-object checks passed.")

,cells,HVGs,donors,cell_types,sites,PCA_dims,Harmony_dims,scVI_dims,has_counts_layer
0,10107,1000,23,13,2,15,15,10,False


All frozen-object checks passed.


## Stage-by-stage audit

In [3]:
stage_df = pd.DataFrame(summary["stages"]).T
stage_df.index.name = "stage"
display(stage_df)
display(pd.DataFrame(summary["expected_checks"]).T)

# The legacy PBMC notebook produced 11,290 healthy/capped cells from 23 donors
# before the benchmark eligibility filter.
assert summary["stages"]["after_downsampling"]["n_cells"] == 11290
assert summary["stages"]["after_downsampling"]["n_donors"] == 23

assert summary["stages"]["after_celltype_support_filter"]["n_cells"] == 10107
assert summary["stages"]["after_celltype_support_filter"]["n_celltypes"] == 13

,n_cells,n_genes,n_donors,n_celltypes
stage,,,,
loaded,62509,16299,119,46
after_dataset_filters,11790,16299,23,41
after_downsampling,11290,16299,23,41
after_celltype_support_filter,10107,16299,23,13
after_gene_filtering,10107,16299,23,13
benchmark_ready,10107,1000,23,13


,expected,observed,ok
n_cells,10107,10107,True
n_donors,23,23,True
n_celltypes,13,13,True
n_hvg,1000,1000,True


## Cell-type support and final composition

In [4]:
support_before = pd.read_csv(QC_DIR / "celltype_support_before_filter.csv")
support_final = pd.read_csv(QC_DIR / "celltype_support_final.csv")
celltype_counts = pd.read_csv(QC_DIR / "celltype_counts_final.csv")

display(support_before)
display(celltype_counts)
print("Final support table:")
display(support_final)

,cell_type,n_cells,n_donors,keep_by_cell_count,keep_by_donor_coverage,keep
0,"CD16-positive, CD56-dim natural killer cell, h...",1582,23,True,True,True
1,"naive thymus-derived CD4-positive, alpha-beta ...",1479,21,True,True,True
2,CD14-positive monocyte,1153,23,True,True,True
3,"central memory CD4-positive, alpha-beta T cell",1115,21,True,True,True
4,"naive thymus-derived CD8-positive, alpha-beta ...",781,21,True,True,True
5,T-helper 22 cell,669,21,True,True,True
6,"effector CD8-positive, alpha-beta T cell",656,20,True,True,True
7,naive B cell,624,23,True,True,True
8,"effector memory CD8-positive, alpha-beta T cell",592,21,True,True,True
9,mature NK T cell,449,22,True,True,True


,cell_type,n_cells
0,"CD16-positive, CD56-dim natural killer cell, h...",1582
1,"naive thymus-derived CD4-positive, alpha-beta ...",1479
2,CD14-positive monocyte,1153
3,"central memory CD4-positive, alpha-beta T cell",1115
4,"naive thymus-derived CD8-positive, alpha-beta ...",781
5,T-helper 22 cell,669
6,"effector CD8-positive, alpha-beta T cell",656
7,naive B cell,624
8,"effector memory CD8-positive, alpha-beta T cell",592
9,mature NK T cell,449


Final support table:


,cell_type,n_cells,n_donors,keep_by_cell_count,keep_by_donor_coverage,keep
0,"CD16-positive, CD56-dim natural killer cell, h...",1582,23,True,True,True
1,"naive thymus-derived CD4-positive, alpha-beta ...",1479,21,True,True,True
2,CD14-positive monocyte,1153,23,True,True,True
3,"central memory CD4-positive, alpha-beta T cell",1115,21,True,True,True
4,"naive thymus-derived CD8-positive, alpha-beta ...",781,21,True,True,True
5,T-helper 22 cell,669,21,True,True,True
6,"effector CD8-positive, alpha-beta T cell",656,20,True,True,True
7,naive B cell,624,23,True,True,True
8,"effector memory CD8-positive, alpha-beta T cell",592,21,True,True,True
9,mature NK T cell,449,22,True,True,True


## Donor/site composition

In [5]:
donor_counts = pd.read_csv(QC_DIR / "donor_counts_final.csv")
display(donor_counts.describe())

site_counts = (
    adata.obs["Site"].astype(str).value_counts()
    .rename_axis("Site").rename("n_cells").reset_index()
)
display(site_counts)

donor_by_site = pd.crosstab(
    adata.obs["patient_id"].astype(str),
    adata.obs["Site"].astype(str),
)
display(donor_by_site)

,n_cells
count,23.000000
mean,439.434783
std,23.426531
min,396.000000
25%,426.500000
50%,446.000000
75%,458.000000
max,468.000000


,Site,n_cells
0,Ncl,5280
1,Cambridge,4827


Site,Cambridge,Ncl
patient_id,,
CV0902,398,0
CV0904,462,0
CV0911,442,0
CV0915,451,0
CV0917,463,0
CV0926,417,0
CV0929,402,0
CV0934,446,0
CV0939,458,0


## Provenance checks

In [6]:
for name in [
    "selected_cells.csv",
    "hvg_genes.csv",
    "resolved_config.yaml",
    "software_versions.json",
]:
    path = QC_DIR / name
    print(name, "OK" if path.exists() else "MISSING")

selected = pd.read_csv(QC_DIR / "selected_cells.csv")
hvgs = pd.read_csv(QC_DIR / "hvg_genes.csv")
assert len(selected) == 10107
assert len(hvgs) == 1000

with (QC_DIR / "resolved_config.yaml").open() as f:
    resolved = yaml.safe_load(f)
assert resolved["support"]["stage"] == "after_downsampling"
assert resolved["downsample"]["sample_full_groups"] is False
assert resolved["counts"]["prepare_from_counts"] is False
assert resolved["scvi"]["enabled"] is False

print("PBMC-specific provenance choices are recorded correctly.")

selected_cells.csv OK
hvg_genes.csv OK
resolved_config.yaml OK
software_versions.json OK
PBMC-specific provenance choices are recorded correctly.


## Confirm cells originate from the source object

In [7]:
if not SOURCE_PATH.exists():
    raise FileNotFoundError(SOURCE_PATH)

source = ad.read_h5ad(SOURCE_PATH, backed="r")
source_names = set(source.obs_names.astype(str))
final_names = set(adata.obs_names.astype(str))
assert final_names.issubset(source_names)
print(f"Verified all {len(final_names):,} frozen cells are present in the source object.")

if hasattr(source, "file") and source.file is not None:
    source.file.close()
if hasattr(adata, "file") and adata.file is not None:
    adata.file.close()

Verified all 10,107 frozen cells are present in the source object.
